# 🛰️ AI 가 배우는 세상을 만든다 — 나만의 시뮬레이터 만들기
### 착지 시뮬레이터를 직접 만들고, 두뇌를 붙이고, 진짜 루나랜더까지

**CONNECT AI LAB · 유튜브 멤버십 라이브** · 2026-10-17

10월 15일, 텍사스 SpaceX 시험장 행사에서 운전자 없는 로드스터 시제품이 램프 끝에서 떨어져 **냉가스(압축 공기) 추진기**로 공중에서 돌아 바퀴로 착지했습니다. 지속 비행이라기보다 짧은 점프에 가깝다는 평가도 있지만, 한 가지는 분명합니다. **날아오르는 것보다 내려앉는 게 어렵다.**

로봇 · 자율주행 · 드론 · 공장 — 이런 두뇌는 현실에서 배우면 비싸고 위험하고 느립니다. 그래서 **시뮬레이터**에서 수천 번 부서지며 배웁니다. 오늘은 그 시뮬레이터를 **직접 만듭니다.**

| 순서 | 할 일 | 시간 |
|---|---|---|
| CELL 01 | 준비 | 1분 |
| CELL 02 | 시뮬레이터 = 다섯 가지 재료 · **나만의 카트폴** 40줄 | 1분 |
| CELL 03 | **착지 시뮬레이터** 만들기 — 추진기 세 개 · 파일 하나 | 1분 |
| CELL 04 | 두뇌 붙이기 — 5강의 DQN 함수 그대로 | 3~5분 |
| CELL 05 | 🌀 회전 점수만 주면? — 리워드 해킹 | 3~5분 |
| CELL 06 | 🧱 **MuJoCo 로 세상 만들기** — XML 카트폴 · 달 중력 | 2분 |
| CELL 07 | 🛬 MuJoCo 착지선 — 다리가 진짜로 닿는다 | 3~5분 |
| CELL 08 | 🌙 진짜 루나랜더 (Gymnasium) — 같은 함수로 착륙 | 3~5분 |
| CELL 09 | 🚗 도전 — 로드스터 모드 | 1분 |
| CELL 10 | 내 시뮬레이터를 세상에 — 파일로 저장 · 공유 | 1분 |

> ▶ **런타임 → 모두 실행** · GPU 필요 없음 · 약 20분. 해설 페이지와 함께 보세요.
> ⚠️ 테슬라가 강화학습을 썼다는 발표는 없습니다. 「이런 착지 문제를 강화학습으로도 풀 수 있다」는 실습입니다.

## CELL 01 · 준비

In [ ]:
!pip install -q swig
!pip install -q "gymnasium[box2d]>=1.0" imageio-ffmpeg
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1

import os, math, time, json, random, base64
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import gymnasium as gym, imageio
from gymnasium import spaces
from gymnasium.utils.env_checker import check_env
import matplotlib, matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont
from IPython.display import HTML, display

글꼴경로 = next((p for p in ['/usr/share/fonts/truetype/nanum/NanumGothicBold.ttf',
                            '/System/Library/Fonts/Supplemental/AppleGothic.ttf'] if os.path.exists(p)), None)
if 글꼴경로:
    matplotlib.font_manager.fontManager.addfont(글꼴경로)
    plt.rcParams['font.family'] = matplotlib.font_manager.FontProperties(fname=글꼴경로).get_name()
plt.rcParams['axes.unicode_minus'] = False
def 글꼴(n):
    try: return ImageFont.truetype(글꼴경로, n)
    except Exception: return ImageFont.load_default()
torch.set_num_threads(1)
def 영상보기(화면들, 파일, fps=30, 너비=480):
    imageio.mimsave(파일, 화면들, fps=fps, macro_block_size=1)
    display(HTML(f'<video width="{너비}" controls autoplay loop muted><source src="data:video/mp4;base64,'
                 f'{base64.b64encode(open(파일, "rb").read()).decode()}" type="video/mp4"></video>'))
print('✅ 준비 완료 · gymnasium', gym.__version__)

## CELL 02 · 시뮬레이터 = 다섯 가지 재료

| 재료 | 질문 | 코드 |
|---|---|---|
| **상태** | 지금 세상은 어떤가? | `observation_space`, `reset()` 이 돌려주는 숫자 |
| **행동** | 무엇을 할 수 있나? | `action_space` |
| **보상** | 무엇이 잘한 것인가? | `step()` 이 돌려주는 숫자 하나 |
| **끝** | 언제 한 판이 끝나나? | `step()` 이 돌려주는 `끝` · `잘림` |
| **그리기** | 눈으로 보려면? | `render()` (오늘은 그림 함수를 따로 씁니다) |

이 다섯 가지를 **Gymnasium 틀**(`gym.Env`)에 맞춰 넣으면, 5강의 DQN 이든 다른 사람이 만든 두뇌든 **누구의 두뇌든 그대로 붙습니다.** 그게 표준의 힘입니다.

먼저 1강부터 써 온 **카트폴을 40줄로 직접** 만들어 봅니다. 물리 공식은 Gymnasium 원본과 같습니다.

In [ ]:
class 나만의카트폴(gym.Env):
    """수레 위 막대 세우기 — 상태 4 · 행동 2 · 버틴 걸음마다 +1"""
    def __init__(self):
        self.action_space = spaces.Discrete(2)                                   # 행동: 0 왼쪽으로 밀기 · 1 오른쪽으로 밀기
        self.observation_space = spaces.Box(-np.inf, np.inf, (4,), np.float32)   # 상태: 수레 위치 · 수레 속도 · 막대 각도 · 막대 각속도
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.s = self.np_random.uniform(-0.05, 0.05, 4); self.t = 0
        return self.s.astype(np.float32), {}
    def step(self, a):
        x, v, th, w = self.s; F_ = 10.0 if a == 1 else -10.0
        c, s_ = math.cos(th), math.sin(th)
        tmp = (F_ + 0.05 * w * w * s_) / 1.1                                      # 물리: 수레 1kg · 막대 0.1kg · 막대 길이 0.5
        ta = (9.8 * s_ - c * tmp) / (0.5 * (4 / 3 - 0.1 * c * c / 1.1)); xa = tmp - 0.05 * ta * c / 1.1
        x, v, th, w = x + 0.02 * v, v + 0.02 * xa, th + 0.02 * w, w + 0.02 * ta
        self.s = np.array([x, v, th, w]); self.t += 1
        끝 = abs(x) > 2.4 or abs(th) > 0.2095                                      # 끝: 밖으로 나가거나 12도 넘게 기울면
        return self.s.astype(np.float32), 1.0, bool(끝), self.t >= 500, {}        # 보상: 버틴 걸음마다 +1

check_env(나만의카트폴())
print('✅ check_env 통과 — Gymnasium 이 「정식 환경」으로 인정했습니다')
e = 나만의카트폴(); 점수 = []
for 씨 in range(20):
    s, _ = e.reset(seed=씨); 끝 = 잘림 = False; R = 0
    while not (끝 or 잘림): s, r, 끝, 잘림, _ = e.step(e.action_space.sample()); R += r
    점수.append(R)
print('🎲 무작위로 20판 →', 점수, '평균', np.mean(점수))

## CELL 03 · 착지 시뮬레이터 만들기

루나랜더를 아주 작게 만든 시뮬레이터입니다. 추진기 세 개로 착지대에 **바퀴부터 사뿐히** 내려앉아야 합니다.

| 재료 | 착지 시뮬레이터에서는 |
|---|---|
| 상태 (6개) | 가로 위치 · 높이 · 가로 속도 · 세로 속도 · 기울기 · 회전 속도 |
| 행동 (4개) | 0 쉬기 · 1 왼쪽 추진기 · 2 가운데 추진기(위로) · 3 오른쪽 추진기 |
| 보상 | 착지 **+100** · 추락 **−100** · 가운데 추진기 한 번 **−0.3** · 옆 추진기 **−0.03** · 착지대에 가까워질수록 · 느려질수록 · 똑바를수록 조금씩 + (다듬기) |
| 끝 | 땅에 닿으면 끝 (착지대 위 · 기울기 0.35 라디안 안 · 천천히 → 착지, 아니면 추락) · 화면 밖 · 400걸음 |

`보상=` 으로 숫자를 바꿀 수 있게 만들었습니다. 오늘 실험의 핵심 손잡이입니다.

In [ ]:
%%writefile 내_착지_시뮬레이터.py
# 시뮬레이터는 파일 하나 — 이 칸이 그 파일을 만듭니다
import math, numpy as np, gymnasium as gym
from gymnasium import spaces

class 착지(gym.Env):

    """추진기 세 개(왼쪽 · 가운데 · 오른쪽)로 착지대에 바퀴부터 사뿐히. 루나랜더를 아주 작게 만든 것."""
    metadata = {'render_modes': ['rgb_array']}
    G, DT, MAIN, SIDE, MAXT = 1.0, 0.05, 2.2, 1.8, 400
    def __init__(self, 모드='기본', 보상=None):
        self.모드 = 모드
        self.P = dict(착지=100.0, 추락=-100.0, 연료_가운데=0.3, 연료_옆=0.03, 다듬기=1.0, 회전=0.0); self.P.update(보상 or {})
        self.action_space = spaces.Discrete(4)    # 0 쉬기 · 1 왼쪽 추진기 · 2 가운데(위로) · 3 오른쪽 추진기
        self.observation_space = spaces.Box(-np.inf, np.inf, (6,), np.float32)
    def _obs(self):
        x, y, vx, vy, th, w = self.s
        return np.array([x / 5, y / 10, vx / 2, vy / 2, th, w / 2], np.float32)
    def _모양(self):
        x, y, vx, vy, th, w = self.s
        return -10 * math.hypot(x / 5, y / 10) - 10 * math.hypot(vx / 2, vy / 2) - 10 * abs(th)
    def reset(self, seed=None, options=None):
        super().reset(seed=seed); r = self.np_random
        th = math.pi if self.모드 == '로드스터' else r.uniform(-0.4, 0.4)
        self.s = np.array([r.uniform(-2.5, 2.5), 10.0, r.uniform(-0.8, 0.8), 0.0, th, r.uniform(-0.3, 0.3)], np.float64)
        self.t = 0; self.prev = self._모양(); self.돈각 = 0.0; return self._obs(), {}
    def step(self, a):
        x, y, vx, vy, th, w = self.s; P = self.P; r = 0.0
        ax, ay, aw = 0.0, -self.G, 0.0
        if a == 2: ax += -math.sin(th) * self.MAIN; ay += math.cos(th) * self.MAIN; r -= P['연료_가운데']
        elif a == 1: aw -= self.SIDE; r -= P['연료_옆']
        elif a == 3: aw += self.SIDE; r -= P['연료_옆']
        vx += ax * self.DT; vy += ay * self.DT; w += aw * self.DT; w *= 0.96
        x += vx * self.DT; y += vy * self.DT; th += w * self.DT; self.돈각 += abs(w * self.DT)
        if self.모드 != '로드스터': th = (th + math.pi) % (2 * math.pi) - math.pi
        self.s = np.array([x, y, vx, vy, th, w]); self.t += 1
        sh = self._모양(); r += P['다듬기'] * (sh - self.prev); self.prev = sh
        r += P['회전'] * abs(w * self.DT)                       # 회전 점수 (로드스터 「묘기」 실험용 — 기본 0)
        끝, 결과 = False, ''
        thn = (th + math.pi) % (2 * math.pi) - math.pi
        if y <= 0:
            끝 = True
            if abs(x) <= 1.5 and abs(thn) < 0.35 and vy > -1.5 and abs(vx) < 1.2: r += P['착지']; 결과 = '착지'
            else: r += P['추락']; 결과 = '추락'
        elif abs(x) > 8 or y > 16: 끝 = True; r += P['추락']; 결과 = '밖으로'
        잘림 = self.t >= self.MAXT and not 끝
        return self._obs(), r, 끝, 잘림, {'결과': 결과 or ('시간 끝' if 잘림 else '')}

In [ ]:
from 내_착지_시뮬레이터 import 착지
check_env(착지())
print('✅ check_env 통과')

def 착지그림(e, a=None, 제목='', 색=(184, 255, 92)):
    W, H, sc = 480, 360, 26; ox, gy = W / 2, H - 40
    im = Image.new('RGB', (W, H), (6, 8, 11)); d = ImageDraw.Draw(im)
    d.rectangle([0, gy, W, H], fill=(30, 34, 42)); d.rectangle([ox - 1.5 * sc, gy - 3, ox + 1.5 * sc, gy + 3], fill=(255, 212, 71))
    for fx in (-1.5, 1.5): d.line([ox + fx * sc, gy, ox + fx * sc, gy - 26], fill=(220, 220, 220), width=2); d.polygon([(ox + fx * sc, gy - 26), (ox + fx * sc + 12, gy - 21), (ox + fx * sc, gy - 16)], fill=(255, 212, 71))
    x, y, vx, vy, th, w = e.s; cx, cy = ox + x * sc, gy - y * sc
    def rot(px, py): c, s = math.cos(th), math.sin(th); return (cx + px * c + py * s, cy - (-px * s + py * c))
    body = [rot(-14, 8), rot(14, 8), rot(14, -6), rot(-14, -6)]; d.polygon(body, fill=(232, 246, 255))
    for wx in (-9, 9): p = rot(wx, -9); d.ellipse([p[0] - 4, p[1] - 4, p[0] + 4, p[1] + 4], fill=(40, 40, 40))
    if a == 2: p1, p2, p3 = rot(-5, -10), rot(5, -10), rot(0, -26); d.polygon([p1, p2, p3], fill=(255, 140, 40))
    if a == 1: p1, p2, p3 = rot(14, 6), rot(14, 0), rot(26, 3); d.polygon([p1, p2, p3], fill=(143, 211, 255))
    if a == 3: p1, p2, p3 = rot(-14, 6), rot(-14, 0), rot(-26, 3); d.polygon([p1, p2, p3], fill=(143, 211, 255))
    d.text((12, 10), 제목, fill=색, font=글꼴(18)); d.text((12, 36), f'높이 {y:4.1f} · 세로속도 {vy:+.2f} · 기울기 {th:+.2f}', fill=(200, 210, 220), font=글꼴(13))
    return np.array(im)

def 착지영상(e, 정책, 씨앗, 제목, 파일, 색=(184, 255, 92)):
    o, _ = e.reset(seed=씨앗); 화면들 = []; 끝 = 잘림 = False
    while not (끝 or 잘림):
        a = 정책(o); o, r, 끝, 잘림, 정보 = e.step(a); 화면들.append(착지그림(e, a, 제목, 색))
    화면들 += [화면들[-1]] * 20; 영상보기(화면들, 파일, fps=30); return 정보['결과']

e = 착지(); 결과 = {}
for 씨 in range(200):
    o, _ = e.reset(seed=씨); 끝 = 잘림 = False
    while not (끝 or 잘림): o, r, 끝, 잘림, 정보 = e.step(e.action_space.sample())
    결과[정보['결과']] = 결과.get(정보['결과'], 0) + 1
print('🎲 무작위로 200판 →', 결과)
print('무작위 영상 →', 착지영상(착지(), lambda o: random.randrange(4), 3, '무작위 조종', 'lander_random.mp4', (255, 93, 93)))

## CELL 04 · 두뇌 붙이기 — 5강의 DQN 함수 그대로

시뮬레이터가 Gymnasium 틀이라 **5강에서 쓴 DQN 함수를 한 글자도 안 바꾸고** 붙입니다. 탐험(2강) · 기억 상자(3강) · γ(4강) · 목표 두뇌(5강).

In [ ]:
def 두뇌만들기(상태수, 행동수, 은닉=128):
    return nn.Sequential(nn.Linear(상태수, 은닉), nn.ReLU(), nn.Linear(은닉, 은닉), nn.ReLU(), nn.Linear(은닉, 행동수))

def DQN학습(환경, 상태수, 행동수, 총걸음, 은닉=128, 학습률=1e-3, 감마=0.99, 탐험기간=0.3, 목표주기=500,
           학습시작=1000, 배치=64, 기억크기=50_000, 씨앗=0, 보고=5000, 평가=None):
    random.seed(씨앗); np.random.seed(씨앗); torch.manual_seed(씨앗)
    q = 두뇌만들기(상태수, 행동수, 은닉); 목표 = 두뇌만들기(상태수, 행동수, 은닉); 목표.load_state_dict(q.state_dict())
    opt = torch.optim.Adam(q.parameters(), lr=학습률)
    S = np.zeros((기억크기, 상태수), np.float32); A = np.zeros(기억크기, np.int64); R = np.zeros(기억크기, np.float32)
    S2 = np.zeros((기억크기, 상태수), np.float32); D = np.zeros(기억크기, np.float32); k = 0; 찬 = 0
    s = 환경.reset(); 판점수 = []; 지금점수 = 0; 최고 = (-1e9, None); t0 = time.time()
    for 걸음 in range(1, 총걸음 + 1):
        eps = max(0.02, 1 - 걸음 / (총걸음 * 탐험기간))
        if random.random() < eps: a = random.randrange(행동수)
        else:
            with torch.no_grad(): a = int(q(torch.from_numpy(s)).argmax())
        s2, r, 끝, 진짜끝 = 환경.step(a)
        S[k], A[k], R[k], S2[k], D[k] = s, a, r, s2, float(진짜끝); k = (k + 1) % 기억크기; 찬 = min(찬 + 1, 기억크기)
        지금점수 += r; s = s2
        if 끝: 판점수.append(지금점수); 지금점수 = 0; s = 환경.reset()
        if 걸음 > 학습시작:
            i = np.random.randint(0, 찬, 배치)
            with torch.no_grad():
                정답 = torch.from_numpy(R[i]) + 감마 * 목표(torch.from_numpy(S2[i])).max(1).values * (1 - torch.from_numpy(D[i]))
            예측 = q(torch.from_numpy(S[i])).gather(1, torch.from_numpy(A[i]).unsqueeze(1)).squeeze(1)
            loss = F.smooth_l1_loss(예측, 정답); opt.zero_grad(); loss.backward(); opt.step()
        if 걸음 % 목표주기 == 0: 목표.load_state_dict(q.state_dict())
        if 걸음 % 보고 == 0:
            줄 = f'[{걸음:>7,}걸음 | {time.time()-t0:4.0f}초 | ε {eps:.2f}] 최근 20판 평균 {np.mean(판점수[-20:]) if 판점수 else 0:7.1f}'
            if 평가:
                점수 = 평가(q); 줄 += f' | 시험 {점수:6.1f}'
                if 점수 >= 최고[0]: 최고 = (점수, {k_: v.clone() for k_, v in q.state_dict().items()}); 줄 += '  🏅'
            print(줄)
    if 평가 and 최고[1] is not None: q.load_state_dict(최고[1])
    return q, 판점수

class 감싸기:                     # Gymnasium 의 5개짜리 돌려주기 → DQN 함수가 쓰는 4개짜리로
    def __init__(self, 만들기): self.e = 만들기(); self.k = 0
    def reset(self): self.k += 1; o, _ = self.e.reset(seed=self.k); return np.asarray(o, np.float32)
    def step(self, a): o, r, 끝, 잘림, _ = self.e.step(a); return np.asarray(o, np.float32), r, 끝 or 잘림, 끝

def 착지율(q, 모드='기본', 보상=None, 판=50):
    e = 착지(모드, 보상); ok = 0
    for 씨 in range(5000, 5000 + 판):
        o, _ = e.reset(seed=씨); 끝 = 잘림 = False
        while not (끝 or 잘림):
            with torch.no_grad(): o, r, 끝, 잘림, 정보 = e.step(int(q(torch.from_numpy(o)).argmax()))
        ok += 정보['결과'] == '착지'
    return ok / 판 * 100

착지두뇌, 착지점수 = DQN학습(감싸기(lambda: 착지()), 6, 4, 총걸음=250_000, 학습률=5e-4, 탐험기간=0.3, 목표주기=500, 배치=128,
                         기억크기=100_000, 보고=31_250, 평가=lambda q: 착지율(q, 판=20))
print(f'\n✅ 다 배운 두뇌 — 처음 보는 50판 착지율 {착지율(착지두뇌):.0f}%  (무작위는 0%)')
print('영상 →', 착지영상(착지(), lambda o: int(착지두뇌(torch.from_numpy(o)).argmax()), 5003, 'DQN 두뇌', 'lander_dqn.mp4'))

## CELL 05 · 🌀 회전 점수만 주면? — 리워드 해킹

로드스터처럼 「공중에서 멋지게 돌기」를 시키고 싶어서, **돌 때마다 점수**(`회전=30`)를 주고 **착지 점수는 없앴습니다**(`착지=0`). 무슨 일이 일어날까요?

> AI 는 **시킨 대로** 합니다. 「댓글 많이 달아」라고만 시키면 아무 댓글이나 답니다. 원하는 것을 보상으로 정확히 말해야 합니다.

In [ ]:
해킹보상 = {'회전': 30.0, '착지': 0.0}
해킹두뇌, 해킹점수 = DQN학습(감싸기(lambda: 착지('기본', 해킹보상)), 6, 4, 총걸음=250_000, 학습률=5e-4, 탐험기간=0.3, 목표주기=500, 배치=128,
                         기억크기=100_000, 보고=31_250)
plt.figure(figsize=(9, 3)); plt.plot(np.convolve(해킹점수, np.ones(20) / 20, 'valid'), color='#ffd447'); plt.title('학습 점수는 높은데 —'); plt.xlabel('판'); plt.show()
print(f'그런데 착지율은 → {착지율(해킹두뇌):.0f}%')
print('영상 →', 착지영상(착지('기본', 해킹보상), lambda o: int(해킹두뇌(torch.from_numpy(o)).argmax()), 5003, '회전 점수만 받은 두뇌', 'lander_hack.mp4', (255, 212, 71)))

## CELL 06 · 🧱 MuJoCo 로 세상 만들기 — XML 이 세상의 설계도

**MuJoCo** 는 딥마인드가 무료로 공개한 물리 엔진입니다(github.com/google-deepmind/mujoco). 로봇 연구의 표준이고, 강화학습 7~8강의 두 발 로봇 · 휴머노이드도 MuJoCo 로 합니다.

MuJoCo 에서는 **XML 몇 줄**이 세상의 설계도입니다. 몸(상자 · 막대 · 다리), 관절(미끄러지기 · 돌기), 질량, 바닥, 중력을 적으면 MuJoCo 가 **진짜 물리**로 움직여 줍니다. 그걸 Gymnasium 틀로 감싸면 — 5강의 DQN 이 그대로 붙습니다.

> 그리기(화면)는 GPU 런타임이면 EGL, CPU 면 OSMesa 로 자동으로 고릅니다. 학습에는 GPU 가 필요 없습니다.

In [ ]:
!pip install -q mujoco
import os, sys, subprocess
if sys.platform == 'darwin': os.environ['MUJOCO_GL'] = 'cgl'
else:
    try: subprocess.check_output('nvidia-smi'); os.environ['MUJOCO_GL'] = 'egl'
    except Exception:
        print('CPU 런타임 — 화면 그리기 도구(OSMesa)를 설치합니다 (1분)…')
        subprocess.run('apt-get -qq update > /dev/null 2>&1; apt-get -qq install -y libosmesa6 libosmesa6-dev libgl1 > /dev/null 2>&1', shell=True)
        os.environ['MUJOCO_GL'] = 'osmesa'; os.environ['PYOPENGL_PLATFORM'] = 'osmesa'
try:
    import mujoco
    _t = mujoco.Renderer(mujoco.MjModel.from_xml_string('<mujoco><worldbody><geom type="sphere" size=".1"/></worldbody></mujoco>'), 32, 32); _t.close()
except Exception as e:
    raise SystemExit('❌ 화면 그리기를 준비하지 못했습니다 → 「런타임 → 런타임 유형 변경 → T4 GPU」 후 「세션 다시 시작」 (' + str(e)[:80] + ')')
print('✅ MuJoCo', mujoco.__version__, '· 그리기', os.environ['MUJOCO_GL'])

카트폴_XML = """
<mujoco model="나만의 카트폴">
  <option timestep="0.01" gravity="0 0 -9.81"/>
  <visual><headlight ambient=".45 .45 .45" diffuse=".6 .6 .6"/></visual>
  <asset><texture name="하늘" type="skybox" builtin="gradient" rgb1=".05 .07 .12" rgb2="0 0 0" width="200" height="200"/>
         <texture name="격자" type="2d" builtin="checker" rgb1=".12 .14 .17" rgb2=".18 .2 .24" width="300" height="300"/>
         <material name="바닥재" texture="격자" texrepeat="8 2"/></asset>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="옆" pos="0 -3.6 1.1" xyaxes="1 0 0 0 0.25 1"/>
    <geom name="바닥" type="plane" size="4 1 0.1" material="바닥재"/>
    <geom name="레일" type="box" pos="0 0 0.5" size="2.4 0.02 0.02" rgba=".5 .5 .5 1" contype="0" conaffinity="0"/>
    <body name="수레" pos="0 0 0.5">
      <joint name="밀기" type="slide" axis="1 0 0" range="-2.4 2.4" limited="true" damping="0.1"/>
      <geom type="box" size="0.2 0.12 0.08" mass="1" rgba=".9 .95 1 1"/>
      <body name="막대" pos="0 0 0.08">
        <joint name="기울기" type="hinge" axis="0 1 0" damping="0.01"/>
        <geom type="capsule" fromto="0 0 0 0 0 1.0" size="0.03" mass="0.1" rgba="1 .83 .28 1"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor joint="밀기" gear="10" ctrlrange="-1 1"/></actuator>
</mujoco>"""
class MJ카트폴(gym.Env):
    def __init__(self, xml=카트폴_XML):
        self.m = mujoco.MjModel.from_xml_string(xml); self.d = mujoco.MjData(self.m)
        self.action_space = spaces.Discrete(2); self.observation_space = spaces.Box(-np.inf, np.inf, (4,), np.float32)
    def _o(self): return np.array([self.d.qpos[0], self.d.qvel[0], self.d.qpos[1], self.d.qvel[1]], np.float32)
    def reset(self, seed=None, options=None):
        super().reset(seed=seed); mujoco.mj_resetData(self.m, self.d)
        self.d.qpos[:] = self.np_random.uniform(-0.05, 0.05, 2); self.d.qvel[:] = self.np_random.uniform(-0.05, 0.05, 2); self.t = 0
        mujoco.mj_forward(self.m, self.d); return self._o(), {}
    def step(self, a):
        self.d.ctrl[0] = 1.0 if a == 1 else -1.0
        for _ in range(2): mujoco.mj_step(self.m, self.d)      # 0.02초
        self.t += 1; o = self._o(); 끝 = bool(abs(o[0]) > 2.3 or abs(o[2]) > 0.21)
        return o, 1.0, 끝, self.t >= 500, {}

def MJ영상(env, 정책, 씨앗, 파일, 제목='', 프레임=600):
    r = mujoco.Renderer(env.m, 360, 640); o, _ = env.reset(seed=씨앗); 화면들 = []; 끝 = 잘림 = False; n = 0
    while not (끝 or 잘림) and n < 프레임:
        a = 정책(o); o, rew, 끝, 잘림, 정보 = env.step(a); n += 1
        r.update_scene(env.d, camera='옆'); im = Image.fromarray(r.render()); d = ImageDraw.Draw(im)
        d.text((14, 10), 제목, fill=(184, 255, 92), font=글꼴(20)); 화면들.append(np.array(im))
    영상보기(화면들[::2], 파일, fps=25, 너비=560); return 정보.get('결과', n)

def MJ시험(q, 만들기=lambda: MJ카트폴(), 판=5):
    e = 만들기(); 합 = 0
    for 씨 in range(1000, 1000 + 판):
        o, _ = e.reset(seed=씨); 끝 = 잘림 = False
        while not (끝 or 잘림):
            with torch.no_grad(): o, rew, 끝, 잘림, _ = e.step(int(q(torch.from_numpy(o)).argmax()))
            합 += rew
    return 합 / 판

e = MJ카트폴(); check_env(e); print('✅ check_env 통과 — XML 로 만든 세상이 정식 Gymnasium 환경')
L = []
for 씨 in range(20):
    o, _ = e.reset(seed=씨); 끝 = 잘림 = False; n = 0
    while not (끝 or 잘림): o, rew, 끝, 잘림, _ = e.step(e.action_space.sample()); n += 1
    L.append(n)
print('🎲 무작위로 20판 → 평균', np.mean(L), '걸음')
MJ카트폴두뇌, _ = DQN학습(감싸기(lambda: MJ카트폴()), 4, 2, 총걸음=50_000, 학습률=1e-3, 탐험기간=0.2, 목표주기=200, 배치=128, 보고=10_000,
                       평가=lambda q: MJ시험(q))
print(f'✅ MuJoCo 카트폴 — 처음 보는 10판 평균 {MJ시험(MJ카트폴두뇌, 판=10):.0f}점 (최대 500)')
print('영상 →', MJ영상(MJ카트폴(), lambda o: int(MJ카트폴두뇌(torch.from_numpy(o)).argmax()), 7, 'mj_cartpole.mp4', 'MuJoCo 카트폴 · DQN 두뇌'), '걸음')

**✏️ 세상을 바꿔 보세요.** XML 숫자 하나면 됩니다.
- 달 중력: `gravity="0 0 -9.81"` → `gravity="0 0 -1.62"`
- 막대 두 배: `fromto="0 0 0 0 0 1.0"` → `fromto="0 0 0 0 0 2.0"`

미리 해 보니 세 세상 모두 같은 DQN 5만 걸음으로 500점이 나왔습니다 — **세상이 바뀌어도 배우는 방법은 같다.** 바꾼 XML 은 `MJ카트폴(새XML)` 로 넣습니다.

In [ ]:
달XML = 카트폴_XML.replace('gravity="0 0 -9.81"', 'gravity="0 0 -1.62"')
print('🌙 달 중력 카트폴 · 무작위 →', MJ영상(MJ카트폴(달XML), lambda o: random.randrange(2), 3, 'mj_moon_random.mp4', '달 중력 · 무작위 (막대가 천천히 쓰러진다)'), '걸음')

## CELL 07 · 🛬 MuJoCo 착지선 — 다리가 진짜로 땅에 닿는다

이번엔 몸통 · 다리 둘 · 착지대를 XML 로 만든 착지선입니다. 직접 짠 착지 시뮬레이터(CELL 03)와 달리 **다리가 땅에 닿는 순간을 MuJoCo 가 진짜 충돌로 계산**합니다. 두 다리가 닿고 멈추면 착지, 몸통이 닿거나 크게 기울면 추락.
진짜 물리에 가까워질수록 배우기도 어렵습니다 — 미리 해 보니 무작위 착지 5% → DQN 25만 걸음 후 48~56%.

In [ ]:
착지선_XML = """
<mujoco model="나만의 착지선">
  <option timestep="0.01" gravity="0 0 -3.0"/>
  <visual><headlight ambient=".45 .45 .45" diffuse=".6 .6 .6"/></visual>
  <asset><texture name="하늘" type="skybox" builtin="gradient" rgb1=".05 .07 .12" rgb2="0 0 0" width="200" height="200"/>
         <texture name="격자" type="2d" builtin="checker" rgb1=".12 .14 .17" rgb2=".18 .2 .24" width="300" height="300"/>
         <material name="바닥재" texture="격자" texrepeat="10 2"/></asset>
  <worldbody>
    <light pos="0 -4 8" dir="0 .5 -1"/>
    <camera name="옆" pos="0 -18 4.5" xyaxes="1 0 0 0 0 1" fovy="50"/>
    <geom name="바닥" type="plane" size="12 2 0.1" material="바닥재"/>
    <geom name="착지대" type="box" pos="0 0 0.05" size="1.5 1 0.05" rgba="1 .83 .28 1"/>
    <body name="착지선" pos="0 0 8">
      <joint name="가로" type="slide" axis="1 0 0"/>
      <joint name="세로" type="slide" axis="0 0 1"/>
      <joint name="기울기" type="hinge" axis="0 1 0"/>
      <geom name="몸통" type="box" size="0.5 0.4 0.35" mass="1" rgba=".9 .95 1 1"/>
      <geom name="왼다리" type="capsule" fromto="-0.4 0 -0.3 -0.8 0 -0.9" size="0.06" mass="0.05" rgba=".6 .65 .7 1"/>
      <geom name="오른다리" type="capsule" fromto="0.4 0 -0.3 0.8 0 -0.9" size="0.06" mass="0.05" rgba=".6 .65 .7 1"/>
    </body>
  </worldbody>
</mujoco>"""
class MJ착지(gym.Env):
    """MuJoCo 물리로 만든 착지선 — 추진기 셋(왼쪽 · 가운데 · 오른쪽), 다리가 진짜로 땅에 닿는다"""
    MAIN, SIDE, MAXT = 5.5, 1.2, 500
    def __init__(self, xml=착지선_XML, 보상=None):
        self.m = mujoco.MjModel.from_xml_string(xml); self.d = mujoco.MjData(self.m)
        self.P = dict(착지=100.0, 추락=-100.0, 연료_가운데=0.3, 연료_옆=0.03); self.P.update(보상 or {})
        self.action_space = spaces.Discrete(4); self.observation_space = spaces.Box(-np.inf, np.inf, (8,), np.float32)
        self.몸 = mujoco.mj_name2id(self.m, mujoco.mjtObj.mjOBJ_BODY, '착지선')
        self.다리 = [mujoco.mj_name2id(self.m, mujoco.mjtObj.mjOBJ_GEOM, n) for n in ('왼다리', '오른다리')]
        self.땅 = {mujoco.mj_name2id(self.m, mujoco.mjtObj.mjOBJ_GEOM, n) for n in ('바닥', '착지대')}
        self.몸통 = mujoco.mj_name2id(self.m, mujoco.mjtObj.mjOBJ_GEOM, '몸통')
    def _닿음(self):
        L = [False, False]; 몸통닿음 = False
        for i in range(self.d.ncon):
            c = self.d.contact[i]; g = {c.geom1, c.geom2}
            if g & self.땅:
                for k, lg in enumerate(self.다리):
                    if lg in g: L[k] = True
                if self.몸통 in g: 몸통닿음 = True
        return L, 몸통닿음
    def _o(self):
        x, z, th = self.d.qpos; vx, vz, w = self.d.qvel; L, _ = self._닿음()
        return np.array([x / 5, (z - 1.0) / 8, vx / 3, vz / 3, th, w / 2, float(L[0]), float(L[1])], np.float32)
    def _모양(self):
        x, z, th = self.d.qpos; vx, vz, w = self.d.qvel
        return -10 * np.hypot(x / 5, (z - 1.0) / 8) - 10 * np.hypot(vx / 3, vz / 3) - 10 * abs(th)
    def reset(self, seed=None, options=None):
        super().reset(seed=seed); mujoco.mj_resetData(self.m, self.d); r = self.np_random
        self.d.qpos[:] = [r.uniform(-3, 3), 0.0, r.uniform(-0.3, 0.3)]; self.d.qvel[:] = [r.uniform(-1, 1), 0.0, r.uniform(-0.3, 0.3)]
        mujoco.mj_forward(self.m, self.d); self.t = 0; self.prev = self._모양(); self.멈춤 = 0; return self._o(), {}
    def step(self, a):
        th = self.d.qpos[2]; P = self.P; r = 0.0; f = np.zeros(6)
        if a == 2: f[0] = np.sin(th) * self.MAIN; f[2] = np.cos(th) * self.MAIN      # 몸이 기운 쪽으로 밀어 올린다; r -= P['연료_가운데']
        elif a == 1: f[4] = -self.SIDE; r -= P['연료_옆']
        elif a == 3: f[4] = self.SIDE; r -= P['연료_옆']
        self.d.xfrc_applied[self.몸] = f
        for _ in range(4): mujoco.mj_step(self.m, self.d)
        self.t += 1; sh = self._모양(); r += sh - self.prev; self.prev = sh
        L, 몸통 = self._닿음(); x, z, th = self.d.qpos; vx, vz, w = self.d.qvel
        끝 = False; 결과 = ''
        if 몸통 or abs(th) > 1.2: 끝 = True; r += P['추락']; 결과 = '추락'
        elif abs(x) > 10: 끝 = True; r += P['추락']; 결과 = '밖으로'
        elif L[0] and L[1]:
            r += 0.5
            if abs(vx) < 0.3 and abs(vz) < 0.3 and abs(w) < 0.3: self.멈춤 += 1
            if self.멈춤 >= 10: 끝 = True; ok = abs(x) <= 1.5; r += P['착지'] if ok else 30; 결과 = '착지' if ok else '착지대 밖 착지'
        잘림 = self.t >= self.MAXT and not 끝
        return self._o(), r, 끝, 잘림, {'결과': 결과 or ('시간 끝' if 잘림 else '')}

check_env(MJ착지()); print('✅ check_env 통과')
def MJ착지율(q, 판=50):
    e = MJ착지(); ok = 0
    for 씨 in range(5000, 5000 + 판):
        o, _ = e.reset(seed=씨); 끝 = 잘림 = False
        while not (끝 or 잘림):
            with torch.no_grad(): o, rew, 끝, 잘림, 정보 = e.step(int(q(torch.from_numpy(o)).argmax()))
        ok += 정보['결과'] == '착지'
    return ok / 판 * 100
MJ착지두뇌, _ = DQN학습(감싸기(lambda: MJ착지()), 8, 4, 총걸음=250_000, 학습률=5e-4, 탐험기간=0.3, 목표주기=500, 배치=128,
                       기억크기=100_000, 보고=31_250, 평가=lambda q: MJ착지율(q, 20))
print(f'\n✅ MuJoCo 착지선 — 처음 보는 50판 착지율 {MJ착지율(MJ착지두뇌):.0f}%')
print('영상 →', MJ영상(MJ착지(), lambda o: int(MJ착지두뇌(torch.from_numpy(o)).argmax()), 5003, 'mj_lander.mp4', 'MuJoCo 착지선 · DQN 두뇌'))

## CELL 08 · 🌙 진짜 루나랜더 — Gymnasium 원본, 같은 함수

이제 Gymnasium 의 `LunarLander-v3`. 물리 엔진(Box2D)이 들어간 진짜 시뮬레이터입니다. 상태 8개(위치 x · y, 속도 x · y, 기울기, 회전 속도, 왼다리 · 오른다리 닿음), 행동 4개.
보상은 착지대로 내려와 멈추면 약 +100~140 · 추락 −100 · 다리가 닿을 때마다 +10 · 가운데 엔진 한 번 −0.3 · 옆 엔진 −0.03. **200점 이상이면 성공.**

In [ ]:
def 루나시험(q, 판=5):
    env = gym.make('LunarLander-v3'); 합 = 0
    for 씨 in range(2000, 2000 + 판):
        s, _ = env.reset(seed=씨); 끝 = False
        while not 끝:
            with torch.no_grad(): a = int(q(torch.from_numpy(s.astype(np.float32))).argmax())
            s, r, t1, t2, _ = env.step(a); 합 += r; 끝 = t1 or t2
    return 합 / 판

루나두뇌, 루나점수 = DQN학습(감싸기(lambda: gym.make('LunarLander-v3')), 8, 4, 총걸음=200_000, 학습률=5e-4, 탐험기간=0.3, 목표주기=500, 배치=128,
                         기억크기=100_000, 보고=20_000, 평가=루나시험)
print(f'\n✅ 처음 보는 10판 평균 {루나시험(루나두뇌, 10):.1f}점  (200 이상 = 성공)')
env = gym.make('LunarLander-v3', render_mode='rgb_array'); s, _ = env.reset(seed=2003); 화면들 = []; 끝 = False
while not 끝:
    with torch.no_grad(): a = int(루나두뇌(torch.from_numpy(s.astype(np.float32))).argmax())
    s, r, t1, t2, _ = env.step(a); 끝 = t1 or t2; 화면들.append(env.render())
영상보기(화면들[::2], 'lunar.mp4', fps=25)

## CELL 09 · 🚗 도전 — 로드스터 모드

`착지('로드스터')` 는 **뒤집힌 채로** 떨어져 공중에서 돌아 바퀴로 착지해야 합니다. 시연에서 본 그 장면이죠.
같은 DQN 으로 60만 걸음을 학습해도 착지율은 **0%** 였습니다(미리 해 본 결과). 돌아야 하는 동시에 천천히 내려야 해서 훨씬 어렵습니다.
행동을 직접 고르는 정책 학습 **PPO**(강화학습 6강)로 도전해 보세요. 아래는 무작위 조종입니다.

In [ ]:
print('로드스터 모드 · 무작위 →', 착지영상(착지('로드스터'), lambda o: random.randrange(4), 7, '로드스터 모드 · 무작위', 'roadster_random.mp4', (255, 93, 93)))

## CELL 10 · 내 시뮬레이터를 세상에

시뮬레이터는 파일 하나입니다. 저장해서 깃허브나 허깅페이스에 올리면 **누구나 내 세상에서 두뇌를 키울 수 있습니다.** 회사의 일 · 매장 · 공장 · 게임 — 무엇이든 다섯 가지 재료로 옮기면 시뮬레이터가 됩니다.

In [ ]:
torch.save(착지두뇌.state_dict(), '착지두뇌.pt')
print('💾 내_착지_시뮬레이터.py (CELL 03 이 만든 파일) · 착지두뇌.pt — 왼쪽 📁 에서 내려받기')
try:
    from google.colab import files; files.download('내_착지_시뮬레이터.py')
except Exception: pass

## 정리 — 시뮬레이터가 사업이 되는 이유

- **두뇌를 만드는 것만큼, 두뇌가 만들어지는 세상(시뮬레이터)을 만드는 것도 중요합니다.**
- 돈이 되는 길: ① 환경을 만들어 패키지로 (Gymnasium 표준) ② 기업 맞춤 시뮬레이터 ③ 대회 · 벤치마크 ④ 학습 데이터 만들기 ⑤ 교육
- 예: **connect-tycoon** — 회사 운영 시뮬레이터 (Connect AI LAB 오픈소스) · github.com/wonseokjung/connect-tycoon
- 오늘 숙제: 내 일 하나를 골라 **상태 · 행동 · 보상 · 끝** 네 줄로 적어 보세요. 그게 내 첫 시뮬레이터 설계도입니다.

> 🔒 이 노트북과 코드는 Connect AI LAB 이 만들었습니다. 공부에는 마음껏, 다른 강의를 만드는 데는 쓰지 말아 주세요.